# Improved UNet SAM2 Experiment

This notebook keeps the original PAN training flow but changes the weak points found in the audit: paper-style 3-class grouping, validated curriculum learning, dynamic class weights, and class-balanced crop training.


## Mount drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## Import packages/install libraries

In [ ]:
!pip install segmentation-models-pytorch

In [ ]:
import cv2
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from pathlib import Path
import torchvision.transforms as T

import albumentations as A
from albumentations.pytorch import ToTensorV2
import segmentation_models_pytorch as smp

## Data Preprocessing

In [ ]:
# Using GPU cuz CPU takes too long
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

In [ ]:
# Define transformation

# Add augmentation for training
train_transformation = A.Compose([
    A.PadIfNeeded(min_height=480, min_width=736, border_mode=cv2.BORDER_REFLECT),
    A.HorizontalFlip(p=0.5),
    A.VerticalFlip(p=0.2),
    A.Affine(
        translate_percent=0.05,
        scale=(0.9, 1.1),
        rotate=0,
        p=0.5,
        interpolation=cv2.INTER_LINEAR,      # for image
        mask_interpolation=cv2.INTER_NEAREST
    ),
    A.HueSaturationValue(p=0.3),
    A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),
    A.ElasticTransform(p=0.2),
    A.RandomBrightnessContrast(p=0.3),
    A.GaussNoise(p=0.2),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

transformation = A.Compose([
    A.PadIfNeeded(min_height=480, min_width=736, border_mode=cv2.BORDER_REFLECT),
    A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
    ToTensorV2()
])

/tmp/ipykernel_815/4180201614.py:17: UserWarning: Argument(s) 'max_holes, max_height, max_width' are not valid for transform CoarseDropout
  A.CoarseDropout(max_holes=8, max_height=32, max_width=32, p=0.3),


### Reusable training components


In [ ]:
# Import reusable helpers from the project tools folder.
# In Colab this path should point to the CSS2 folder in Drive.
import sys

PROJECT_ROOT = Path("/content/drive/MyDrive/CSS2")
sys.path.append(str(PROJECT_ROOT / "tools"))

from training_components import (
    GingivitisSegmentationDataset,
    CLASS_NAMES,
    SegmentationMeter,
    compute_class_weights,
)

# Main next experiment from the team discussion.
# Options: "severity_3", "paper_3", "binary_disease", "severity_5".
MODE = "paper_3"
DATA_ROOT = PROJECT_ROOT / "CSS2_480p" / "Dataset"
NUM_CLASSES = len(CLASS_NAMES[MODE])

print("Experiment mode:", MODE)
print("Classes:", CLASS_NAMES[MODE])


Experiment mode: paper_3
Classes: ['healthy', 'questionable', 'diseased']


### Create dataset objects

In [ ]:
train_dataset = GingivitisSegmentationDataset(
        image_folder=DATA_ROOT / "Training" / "Images",
        mask_folder=DATA_ROOT / "Training" / "SAM2_Morph_MasksIndex",
        transform=train_transformation,
        mode=MODE,
        crop_size=384,
        balanced_crop=True,
)

val_dataset = GingivitisSegmentationDataset(
    image_folder=DATA_ROOT / "Validation" / "Images",
    mask_folder=DATA_ROOT / "Validation" / "SAM2_Morph_MasksIndex",
    transform=transformation,
    mode=MODE,
)

test_dataset = GingivitisSegmentationDataset(
    image_folder=DATA_ROOT / "Test" / "Images",
    mask_folder=DATA_ROOT / "Test" / "SAM2_Morph_MasksIndex",
    transform=transformation,
    mode=MODE,
)

print("Train / val / test:", len(train_dataset), len(val_dataset), len(test_dataset))

Train / val / test: 732 182 182


## Create DataLoaders

In [ ]:
train_loader = DataLoader(train_dataset, batch_size=2, shuffle=True, num_workers=2, pin_memory= True) # shuffle is true for training so model learns instead of memorising; pin_memory is to transfer from cpu to gpu faster
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=2, pin_memory= True)
test_loader  = DataLoader(test_dataset, batch_size=2, shuffle=False, num_workers=2, pin_memory= True)

## Load UNET

In [ ]:
model = smp.Unet(
    encoder_name="efficientnet-b0",
    encoder_weights="imagenet",
    in_channels=3,
    classes=NUM_CLASSES
)

model = model.to(device)

# Freeze encoder for first 7 epochs
for param in model.encoder.parameters():
    param.requires_grad = False

In [ ]:
# Loss function and optimizer
weights = compute_class_weights(
    DATA_ROOT / "Training" / "SAM2_Morph_MasksIndex",
    mode=MODE,
    device=device,
)
print("Class weights:")
for name, weight in zip(CLASS_NAMES[MODE], weights.detach().cpu().tolist()):
    print(f"  {name}: {weight:.3f}")

optimizer = torch.optim.Adam(model.parameters(), lr=1e-4, weight_decay=1e-3)


Class weights:
  healthy: 2.018
  questionable: 0.545
  diseased: 0.437


## Training

In [ ]:
num_epochs = 40
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=0.5, patience=5, min_lr=1e-6
)
best_val_loss = float('inf')
patience = 12
patience_counter = 0
ce_loss = torch.nn.CrossEntropyLoss(weight=weights, ignore_index=255)
dice_loss = smp.losses.DiceLoss(mode='multiclass', ignore_index=255)

for epoch in range(num_epochs):

  # Unfreeze encoder after epoch
  if epoch == 7:
      for param in model.encoder.parameters():
          param.requires_grad = True
      print('Encoder unfrozen')

  # Training loop
  model.train()
  total_loss = 0
  train_batches = 0

  for i, (images, masks, names) in enumerate(train_loader):
    images = images.to(device)   # using GPU
    masks = masks.to(device)

    # Skip batch if there are no valid pixels in the mask for loss calculation
    if not (masks != 255).any():
      print(f"Training Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255)")
      continue

    outputs = model(images)
    loss = ce_loss(outputs, masks) + dice_loss(outputs, masks)

    optimizer.zero_grad() # clear old graidents from previous steps
    loss.backward() # backward pass to calculate the gradients for all parameters
    optimizer.step() # update model

    total_loss += loss.item()
    train_batches += 1

  # Validation loop
  model.eval() # set model to evaluation mode
  val_loss = 0
  val_batches = 0

  with torch.no_grad():
    for i, (images, masks, names) in enumerate(val_loader):
      images = images.to(device)
      masks = masks.to(device)

      # Skip batch if there are no valid pixels in the mask for loss calculation
      if not (masks != 255).any():
        print(f"Validation Skipping Batch {i}, images: {names} since all mask pixels are ignore_index (255).")
        continue # Skip this batch

      outputs = model(images)
      loss = ce_loss(outputs, masks) + dice_loss(outputs, masks)
      val_loss += loss.item()
      val_batches += 1

  train_loss = total_loss / train_batches if train_batches > 0 else 0
  val_loss = val_loss / val_batches if val_batches > 0 else 0

  scheduler.step(val_loss)

  print(f"Epoch {epoch+1}")
  print(f"Train loss: {train_loss:.2f}")
  print(f"Val loss: {val_loss:.2f}")

  if val_loss < best_val_loss:
      best_val_loss = val_loss
      patience_counter = 0
      torch.save(model.state_dict(), '/content/best_model.pth')
      torch.save(model.state_dict(), '/content/drive/MyDrive/CSS2/Final Models (PAN, UNet, UNet++, DeepLabV3+)/best_unet.pth')
      print('Improvement')
  else:
      patience_counter += 1
      print(f'No improvement ({patience_counter}/{patience})')
      if patience_counter >= patience:
          print('Early stopping triggered')
          break

Epoch 1
Train loss: 1.14
Val loss: 1.36
Improvement
Epoch 2
Train loss: 0.98
Val loss: 1.30
Improvement
Epoch 3
Train loss: 0.93
Val loss: 1.23
Improvement
Epoch 4
Train loss: 0.92
Val loss: 1.25
No improvement (1/12)
Epoch 5
Train loss: 0.90
Val loss: 1.23
Improvement
Epoch 6
Train loss: 0.89
Val loss: 1.24
No improvement (1/12)
Epoch 7
Train loss: 0.89
Val loss: 1.27
No improvement (2/12)
Encoder unfrozen
Epoch 8
Train loss: 0.89
Val loss: 1.27
No improvement (3/12)
Epoch 9
Train loss: 0.87
Val loss: 1.22
Improvement
Epoch 10
Train loss: 0.87
Val loss: 1.38
No improvement (1/12)
Epoch 11
Train loss: 0.84
Val loss: 1.38
No improvement (2/12)
Epoch 12
Train loss: 0.86
Val loss: 1.35
No improvement (3/12)
Epoch 13
Train loss: 0.83
Val loss: 1.38
No improvement (4/12)
Epoch 14
Train loss: 0.84
Val loss: 1.47
No improvement (5/12)
Epoch 15
Train loss: 0.85
Val loss: 1.46
No improvement (6/12)
Epoch 16
Train loss: 0.80
Val loss: 1.41
No improvement (7/12)
Epoch 17
Train loss: 0.80
Val loss

## Testing

IoU checks how much the prediction overlaps with the ground truth (correct labels).  
Dice measures similarity between prediction and ground truth but gives more weight to overlapping pixels.  
Calculating the mean for these two to show one score overall for them.

In [ ]:
def evaluate(model, dataloader, device, num_classes=3):
  model.load_state_dict(torch.load('/content/best_model.pth'))
  model.eval()

  total_correct = 0
  total_pixels = 0

  # Accumulate per-class stats (dataset-level)
  intersection_per_class = [0] * num_classes
  union_per_class = [0] * num_classes
  dice_num = [0] * num_classes
  dice_den = [0] * num_classes
  tp_per_class = [0] * num_classes
  fn_per_class = [0] * num_classes

  with torch.no_grad():
    for images, masks, names in dataloader:
      images = images.to(device)
      masks = masks.to(device)

      outputs = model(images)
      preds = torch.argmax(outputs, dim=1)

      valid = masks != 255  # ignore background

      # Accuracy
      correct = (preds == masks) & valid
      total_correct += correct.sum().item()
      total_pixels += valid.sum().item()

      # Per-class metrics
      for cls in range(num_classes):
        pred_cls = (preds == cls) & valid
        mask_cls = (masks == cls) & valid

        intersection = (pred_cls & mask_cls).sum().item()
        union = (pred_cls | mask_cls).sum().item()

        # IoU
        intersection_per_class[cls] += intersection
        union_per_class[cls] += union

        # Dice
        dice_num[cls] += 2 * intersection
        dice_den[cls] += pred_cls.sum().item() + mask_cls.sum().item()

        # Recall (TP and FN)
        tp_per_class[cls] += intersection
        fn_per_class[cls] += (mask_cls & (~pred_cls)).sum().item()

  # Accuracy
  accuracy = total_correct / total_pixels if total_pixels > 0 else 0

  # Mean IoU
  iou_scores = []
  for c in range(num_classes):
    if union_per_class[c] > 0:
      iou_scores.append(intersection_per_class[c] / union_per_class[c])
  mean_iou = sum(iou_scores) / len(iou_scores) if iou_scores else 0

  # Mean Dice
  dice_scores = []
  for c in range(num_classes):
    if dice_den[c] > 0:
      dice_scores.append(dice_num[c] / (dice_den[c] + 1e-6))
  mean_dice = sum(dice_scores) / len(dice_scores) if dice_scores else 0

  # Mean Recall
  recall_scores = []
  for c in range(num_classes):
    tp = tp_per_class[c]
    fn = fn_per_class[c]
    if (tp + fn) > 0:
      recall_scores.append(tp / (tp + fn))
  mean_recall = sum(recall_scores) / len(recall_scores) if recall_scores else 0

  return accuracy, mean_iou, mean_dice, mean_recall

In [ ]:
test_acc, test_iou, test_dice, test_recall = evaluate(model, test_loader, device, num_classes=NUM_CLASSES)

print(f"Test Accuracy: {test_acc:.2f}")
print(f"Mean IoU: {test_iou:.2f}")
print(f"Mean Dice: {test_dice:.2f}")
print(f"Mean Recall: {test_recall:.2f}")


Test Accuracy: 0.59
Mean IoU: 0.27
Mean Dice: 0.38
Mean Recall: 0.39


## TTA results


In [ ]:
def predict_tta(model, image, device):
    """Run inference with multiple augmentations and average predictions."""
    model.eval()
    preds = []

    with torch.no_grad():
        # Original
        out = torch.softmax(model(image.to(device)), dim=1)
        preds.append(out)

        # Horizontal flip
        flipped = torch.flip(image, [-1])
        out = torch.softmax(model(flipped.to(device)), dim=1)
        out = torch.flip(out, [-1])  # flip back
        preds.append(out)

        # Vertical flip
        flipped = torch.flip(image, [-2])
        out = torch.softmax(model(flipped.to(device)), dim=1)
        out = torch.flip(out, [-2])  # flip back
        preds.append(out)

    # Average all predictions
    return torch.mean(torch.stack(preds), dim=0)


def evaluate_tta(model, dataloader, device, num_classes=3):
    model.load_state_dict(torch.load('/content/best_model.pth'))
    model.eval()

    total_correct = 0
    total_pixels = 0
    intersection_per_class = [0] * num_classes
    union_per_class = [0] * num_classes
    dice_num = [0] * num_classes
    dice_den = [0] * num_classes
    tp_per_class = [0] * num_classes
    fn_per_class = [0] * num_classes

    with torch.no_grad():
        for images, masks, names in dataloader:
            masks = masks.to(device)

            # Use TTA instead of normal forward pass
            outputs = predict_tta(model, images, device)
            preds = torch.argmax(outputs, dim=1)

            valid = masks != 255

            correct = (preds == masks) & valid
            total_correct += correct.sum().item()
            total_pixels += valid.sum().item()

            for cls in range(num_classes):
                pred_cls = (preds == cls) & valid
                mask_cls = (masks == cls) & valid
                intersection = (pred_cls & mask_cls).sum().item()
                union = (pred_cls | mask_cls).sum().item()
                intersection_per_class[cls] += intersection
                union_per_class[cls] += union
                dice_num[cls] += 2 * intersection
                dice_den[cls] += pred_cls.sum().item() + mask_cls.sum().item()
                tp_per_class[cls] += intersection
                fn_per_class[cls] += (mask_cls & (~pred_cls)).sum().item()

    accuracy = total_correct / total_pixels if total_pixels > 0 else 0

    iou_scores = []
    for c in range(num_classes):
        if union_per_class[c] > 0:
            iou_scores.append(intersection_per_class[c] / union_per_class[c])
    mean_iou = sum(iou_scores) / len(iou_scores) if iou_scores else 0

    dice_scores = []
    for c in range(num_classes):
        if dice_den[c] > 0:
            dice_scores.append(dice_num[c] / (dice_den[c] + 1e-6))
    mean_dice = sum(dice_scores) / len(dice_scores) if dice_scores else 0

    recall_scores = []
    for c in range(num_classes):
        tp = tp_per_class[c]
        fn = fn_per_class[c]
        if (tp + fn) > 0:
            recall_scores.append(tp / (tp + fn))
    mean_recall = sum(recall_scores) / len(recall_scores) if recall_scores else 0

    return accuracy, mean_iou, mean_dice, mean_recall


# Run TTA evaluation
test_acc, test_iou, test_dice, test_recall = evaluate_tta(model, test_loader, device, num_classes=NUM_CLASSES)
print(f"TTA Test Accuracy: {test_acc:.2f}")
print(f"TTA Mean IoU:      {test_iou:.2f}")
print(f"TTA Mean Dice:     {test_dice:.2f}")
print(f"TTA Mean Recall:   {test_recall:.2f}")

TTA Test Accuracy: 0.60
TTA Mean IoU:      0.28
TTA Mean Dice:     0.39
TTA Mean Recall:   0.40


TTA no change means that mask quality is issue here.

## Class imbalance is main issue

In [ ]:
# See per-class IoU breakdown with the active class mapping.
def evaluate_per_class(model, dataloader, device, num_classes=NUM_CLASSES):
    model.load_state_dict(torch.load("/content/best_model.pth"))
    model.eval()

    intersection_per_class = [0] * num_classes
    union_per_class = [0] * num_classes
    support_per_class = [0] * num_classes

    with torch.no_grad():
        for images, masks, _ in dataloader:
            images = images.to(device)
            masks = masks.to(device)
            outputs = model(images)
            preds = torch.argmax(outputs, dim=1)
            valid = masks != 255

            for cls in range(num_classes):
                pred_cls = (preds == cls) & valid
                mask_cls = (masks == cls) & valid
                intersection_per_class[cls] += (pred_cls & mask_cls).sum().item()
                union_per_class[cls] += (pred_cls | mask_cls).sum().item()
                support_per_class[cls] += mask_cls.sum().item()

    class_names = CLASS_NAMES[MODE]
    for c in range(num_classes):
        iou = intersection_per_class[c] / union_per_class[c] if union_per_class[c] > 0 else 0
        print(f"Class {c} ({class_names[c]}): IoU = {iou:.3f}, support = {support_per_class[c]:,} px")

evaluate_per_class(model, test_loader, device)


Class 0 (healthy): IoU = 0.000, support = 103,012 px
Class 1 (questionable): IoU = 0.491, support = 3,653,198 px
Class 2 (diseased): IoU = 0.328, support = 3,121,508 px
